In [1]:
#!pip install openmeteo-requests
#!pip install requests-cache retry-requests numpy pandas

In [2]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 48.85341,
	"longitude": 2.3488,
	"daily": ["temperature_2m_max", "temperature_2m_min", "uv_index_max", "precipitation_hours"],
	"timezone": "Europe/Berlin",
	"forecast_days": 16
}
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation {response.Elevation()} m asl")
print(f"Timezone {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0 {response.UtcOffsetSeconds()} s")

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_temperature_2m_max = daily.Variables(0).ValuesAsNumpy()
daily_temperature_2m_min = daily.Variables(1).ValuesAsNumpy()
daily_uv_index_max = daily.Variables(2).ValuesAsNumpy()
daily_precipitation_hours = daily.Variables(3).ValuesAsNumpy()

daily_data = {"date": pd.date_range(
	start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
	end = pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = daily.Interval()),
	inclusive = "left"
)}

daily_data["temperature_2m_max"] = daily_temperature_2m_max
daily_data["temperature_2m_min"] = daily_temperature_2m_min
daily_data["uv_index_max"] = daily_uv_index_max
daily_data["precipitation_hours"] = daily_precipitation_hours

daily_dataframe = pd.DataFrame(data = daily_data)
print(daily_dataframe)

Coordinates 48.86000061035156°N 2.3399996757507324°E
Elevation 43.0 m asl
Timezone b'Europe/Berlin'b'GMT+2'
Timezone difference to GMT+0 7200 s
                        date  temperature_2m_max  temperature_2m_min  \
0  2025-07-05 22:00:00+00:00           20.819500           14.719500   
1  2025-07-06 22:00:00+00:00           19.369501           15.869500   
2  2025-07-07 22:00:00+00:00           21.104000           14.219500   
3  2025-07-08 22:00:00+00:00           24.304001           11.704000   
4  2025-07-09 22:00:00+00:00           25.704000           14.954000   
5  2025-07-10 22:00:00+00:00           27.403999           16.454000   
6  2025-07-11 22:00:00+00:00           28.504000           16.604000   
7  2025-07-12 22:00:00+00:00           33.618999           17.604000   
8  2025-07-13 22:00:00+00:00           34.218998           23.419001   
9  2025-07-14 22:00:00+00:00           25.719000           20.019001   
10 2025-07-15 22:00:00+00:00           29.019001           17.11

In [3]:
daily_dataframe

,date,temperature_2m_max,temperature_2m_min,uv_index_max,precipitation_hours
0,2025-07-05 22:00:00+00:00,20.819500,14.719500,3.00,15.0
1,2025-07-06 22:00:00+00:00,19.369501,15.869500,3.60,10.0
2,2025-07-07 22:00:00+00:00,21.104000,14.219500,7.10,2.0
3,2025-07-08 22:00:00+00:00,24.304001,11.704000,7.15,0.0
4,2025-07-09 22:00:00+00:00,25.704000,14.954000,7.20,0.0
5,2025-07-10 22:00:00+00:00,27.403999,16.454000,6.90,0.0
6,2025-07-11 22:00:00+00:00,28.504000,16.604000,6.75,0.0
7,2025-07-12 22:00:00+00:00,33.618999,17.604000,6.85,0.0
8,2025-07-13 22:00:00+00:00,34.218998,23.419001,7.00,0.0
9,2025-07-14 22:00:00+00:00,25.719000,20.019001,3.45,0.0


In [4]:
daily_dataframe['date'] = daily_dataframe['date'].dt.strftime('%Y-%m-%d')

In [7]:
daily_dataframe.to_csv('daily_meteo.csv', index=False, encoding='utf-8')